In [2]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
import matplotlib.pyplot as plt
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

Define simulation parameters:

In [3]:
param_table = pd.read_csv('../data_clean/simulation_parameters.csv')

NUM_ITERATIONS = 1000
SCENARIOS      = ['Normal', 'Moderate', 'Severe']
TCWS_COLS      = ['prob_signal1', 'prob_signal2', 'prob_signal3', 'prob_signal4', 'prob_signal5']

SUSPENSION_DAYS = {
    1: [0, 1, 2],
    2: [1, 2, 3],
    3: [2, 3, 4],
    4: [3, 4, 5],
    5: [4, 5, 6]
}

RISK_THRESHOLDS = {
    'Low':      (0,  5),
    'Moderate': (6,  10),
    'High':     (11, 20),
    'Critical': (21, float('inf'))
}

def classify_risk(days_lost):
    if days_lost <= 5:
        return 'Low'
    elif days_lost <= 10:
        return 'Moderate'
    elif days_lost <= 20:
        return 'High'
    else:
        return 'Critical'

print('Constants defined.')
print(f'Iterations: {NUM_ITERATIONS:,}')
print(f'Scenarios: {SCENARIOS}')
print(f'CPUs available: {cpu_count()}')

Constants defined.
Iterations: 1,000
Scenarios: ['Normal', 'Moderate', 'Severe']
CPUs available: 8


In [4]:
def simulate_region_scenario(params):
    region   = params['region']
    scenario = params['scenario']
    lam      = params['lambda']

    tcws_probs = np.array([
        params['prob_signal1'], params['prob_signal2'], params['prob_signal3'],
        params['prob_signal4'], params['prob_signal5']
    ])

    count_low      = 0
    count_moderate = 0
    count_high     = 0
    count_critical = 0

    tcws_probs = tcws_probs / tcws_probs.sum()
    tcws_levels = [1, 2, 3, 4, 5]

    rng = np.random.default_rng()

    desc = f'{region[:20]} | {scenario}'
    for _ in tqdm(range(NUM_ITERATIONS), desc=desc, leave=True):

        num_typhoons    = rng.poisson(lam)
        total_days_lost = 0

        for _ in range(num_typhoons):
            signal          = rng.choice(tcws_levels, p=tcws_probs)
            days            = rng.choice(SUSPENSION_DAYS[signal])
            total_days_lost += days

        risk = classify_risk(total_days_lost)
        if risk == 'Low':
            count_low += 1
        elif risk == 'Moderate':
            count_moderate += 1
        elif risk == 'High':
            count_high += 1
        else:
            count_critical += 1

    return {
        'region':     region,
        'scenario':   scenario,
        'p_low':      round(count_low      / NUM_ITERATIONS, 4),
        'p_moderate': round(count_moderate / NUM_ITERATIONS, 4),
        'p_high':     round(count_high     / NUM_ITERATIONS, 4),
        'p_critical': round(count_critical / NUM_ITERATIONS, 4)
    }

In [5]:
import time

param_list = param_table.to_dict(orient='records')

print(f'Running {len(param_list)} region-scenario simulations...')
print(f'Iterations per simulation: {NUM_ITERATIONS:,}')

start_time = time.time()

results = [simulate_region_scenario(p) for p in param_list]

end_time = time.time()
elapsed  = round(end_time - start_time, 2)

print(f'Simulation completed in {elapsed} seconds.')
print(f'Total results: {len(results)} region-scenario pairs')

Running 51 region-scenario simulations...
Iterations per simulation: 1,000


Bangsamoro Autonomou | Normal:   0%|          | 0/1000 [00:00<?, ?it/s]

Bangsamoro Autonomou | Normal: 100%|██████████| 1000/1000 [00:00<00:00, 50524.04it/s]
Bangsamoro Autonomou | Moderate: 100%|██████████| 1000/1000 [00:00<00:00, 32943.27it/s]
Bangsamoro Autonomou | Severe: 100%|██████████| 1000/1000 [00:00<00:00, 11079.39it/s]
Cordillera Administr | Normal: 100%|██████████| 1000/1000 [00:00<00:00, 3324.24it/s]
Cordillera Administr | Moderate: 100%|██████████| 1000/1000 [00:00<00:00, 2970.41it/s]
Cordillera Administr | Severe: 100%|██████████| 1000/1000 [00:00<00:00, 2766.06it/s]
Mimaropa Region | Normal: 100%|██████████| 1000/1000 [00:00<00:00, 5760.17it/s]
Mimaropa Region | Moderate: 100%|██████████| 1000/1000 [00:00<00:00, 4659.44it/s]
Mimaropa Region | Severe: 100%|██████████| 1000/1000 [00:00<00:00, 3927.14it/s]
National Capital Reg | Normal: 100%|██████████| 1000/1000 [00:00<00:00, 8935.55it/s]
National Capital Reg | Moderate: 100%|██████████| 1000/1000 [00:00<00:00, 6953.10it/s]
National Capital Reg | Severe: 100%|██████████| 1000/1000 [00:00<00:0

Simulation completed in 8.94 seconds.
Total results: 51 region-scenario pairs


In [6]:
results_df = pd.DataFrame(results)


results_df = results_df.sort_values(
    ['scenario', 'region']
).reset_index(drop=True)

prob_cols = ['p_low', 'p_moderate', 'p_high', 'p_critical']
results_df['prob_sum'] = results_df[prob_cols].sum(axis=1)

anomalies = results_df[~results_df['prob_sum'].between(0.99, 1.01)]
print('Probability sum anomalies:', len(anomalies))
print()

results_df = results_df.drop(columns=['prob_sum'])

print('Results sample:')
print(results_df.head(15).to_string(index=False))

results_df.to_csv('../data_clean/simulation_results.csv', index=False)

Probability sum anomalies: 0

Results sample:
                                                 region scenario  p_low  p_moderate  p_high  p_critical
Bangsamoro Autonomous Region In Muslim Mindanao (BARMM) Moderate  0.983       0.015   0.002       0.000
                 Cordillera Administrative Region (CAR) Moderate  0.076       0.175   0.457       0.292
                                        Mimaropa Region Moderate  0.249       0.301   0.377       0.073
                          National Capital Region (NCR) Moderate  0.441       0.300   0.239       0.020
                               Region I (Ilocos Region) Moderate  0.108       0.166   0.443       0.283
                             Region II (Cagayan Valley) Moderate  0.028       0.071   0.371       0.530
                             Region III (Central Luzon) Moderate  0.099       0.218   0.457       0.226
                               Region IV-A (Calabarzon) Moderate  0.180       0.293   0.408       0.119
                  